# Epileptic Seizure Detection — patient-level split (no leakage)

Run the cells from top to bottom. Every model is built here, so every parameter is visible and editable.
Optuna is optional: use `best_param(...)` then `build(**best)` (on), or type the parameters yourself (off).

## 0. Setup (Kaggle)
Upload the project files as a Kaggle **Dataset** (or attach the zip), then set `PROJECT_DIR` and `CSV_PATH` below.

In [ ]:
import sys, os

# ── EDIT THESE TWO LINES ─────────────────────────────────────────────
PROJECT_DIR = '/kaggle/working'                     # folder that contains base.py, models/, etc.
CSV_PATH    = '/kaggle/input/YOUR-DATASET/balanced_dataset8.csv'   # your CSV file
# ─────────────────────────────────────────────────────────────────────

sys.path.insert(0, PROJECT_DIR)
print('Project files found:', sorted(f for f in os.listdir(PROJECT_DIR) if f.endswith('.py')))
print('CSV exists         :', os.path.exists(CSV_PATH))

In [ ]:
# Only needed if a package is missing (Kaggle already has most of them).
# !pip install -q xgboost optuna tqdm

## 1. Imports

In [ ]:
from preprocessing import prepare_data
from models import LogisticModel, XGBoostModel, MambaModel
from experiment import train, evaluate
from report import report, compare
from plots import plot, plot_compare

## 2. Data
`X`, `y`, `g` are dictionaries with keys `'train'`, `'validation'`, `'test'`.  
`g` = patient id of every row (used for patient-grouped tuning and per-patient scores).

In [ ]:
X, y, g = prepare_data(CSV_PATH)

## 3. Logistic Regression — WITHOUT Optuna (you choose the parameters)

In [ ]:
model = LogisticModel.build(C=0.1, penalty='l2', max_iter=300)     # edit any value here
model = train(model, X['train'], y['train'])
scores_lr = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_lr); plot(scores_lr)

## 4. XGBoost — WITHOUT Optuna (all parameters typed by hand)

In [ ]:
model = XGBoostModel.build(n_estimators=500, max_depth=8, learning_rate=0.05,
                           subsample=0.9, colsample_bytree=0.6, min_child_weight=10,
                           gamma=0.0, reg_alpha=0.1, reg_lambda=1.0)
model = train(model, X['train'], y['train'])
scores_xgb = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_xgb); plot(scores_xgb)

## 5. XGBoost — WITH Optuna
`best_param` prints the best parameters; `build(**best)` uses them. Start with a small `n_trials` to check it runs.

In [ ]:
best  = XGBoostModel.best_param(X['train'], y['train'], g['train'], n_trials=10)
model = XGBoostModel.build(**best)
model = train(model, X['train'], y['train'])
scores_xgb_tuned = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
scores_xgb_tuned['name'] = 'XGBoost (Optuna)'
report(scores_xgb_tuned); plot(scores_xgb_tuned)

## 6. Mamba (optional — needs PyTorch; skip this section if you do not use it)
The 22 EEG channels form the sequence. State this in the paper (channel-axis modelling; the CSV has no time order).

In [ ]:
# WITHOUT Optuna:
model = MambaModel.build(d_model=64, n_layers=2, d_state=16, dropout=0.2,
                         lr=1e-3, weight_decay=1e-2, epochs=30)
model = train(model, X['train'], y['train'])
scores_mamba = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_mamba); plot(scores_mamba)

In [ ]:
# WITH Optuna (slow — start with n_trials=3). Tuning runs 10 epochs; final training uses 30.
best  = MambaModel.best_param(X['train'], y['train'], g['train'], n_trials=3)
model = MambaModel.build(**{**best, 'epochs': 30})
model = train(model, X['train'], y['train'])
scores_mamba_tuned = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
scores_mamba_tuned['name'] = 'Mamba (Optuna)'
report(scores_mamba_tuned)

## 7. Compare all models
Only list models you actually ran.

In [ ]:
all_scores = [scores_lr, scores_xgb, scores_xgb_tuned]          # add scores_mamba if you ran it
compare(all_scores); plot_compare(all_scores)

## 8. Wrong-parameter check (optional demo)
Each model rejects parameters that belong to another model.

In [ ]:
try:
    XGBoostModel.build(d_model=64)          # d_model is a Mamba parameter
except ValueError as e:
    print(e)